In [1]:
import sys
import os

spark_home = "/opt/spark"
sys.path.append(os.path.join(spark_home, "python"))
sys.path.append(os.path.join(spark_home, "python/lib/py4j-0.10.9.7-src.zip"))

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSession").getOrCreate()
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 14:19:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
## Schemas

from pyspark.sql.types import StructField, StructType, IntegerType, StringType

schema = StructType([
    StructField("LatD", IntegerType(), True),
    StructField("LatM", IntegerType(), True),
    StructField("LatS", IntegerType(), True),
    StructField("NS", StringType(), True),
    StructField("LonD", IntegerType(), True),
    StructField("LonM", IntegerType(), True),
    StructField("LonS", IntegerType(), True),
    StructField("EW", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True)
])

df = spark.read.format("csv").option("header", "true").option("ignoreLeadingWhiteSpace", "true").option("inferSchema", "false").schema(schema).load("data/cities.csv")
df.limit(5).show()

df.printSchema()
df.schema

+----+----+----+---+----+----+----+---+---------------+-----+
|LatD|LatM|LatS| NS|LonD|LonM|LonS| EW|           City|State|
+----+----+----+---+----+----+----+---+---------------+-----+
|  41|   5|  59|  N|  80|  39|   0|  W|     Youngstown|   OH|
|  42|  52|  48|  N|  97|  23|  23|  W|        Yankton|   SD|
|  46|  35|  59|  N| 120|  30|  36|  W|         Yakima|   WA|
|  42|  16|  12|  N|  71|  48|   0|  W|      Worcester|   MA|
|  43|  37|  48|  N|  89|  46|  11|  W|Wisconsin Dells|   WI|
+----+----+----+---+----+----+----+---+---------------+-----+

root
 |-- LatD: integer (nullable = true)
 |-- LatM: integer (nullable = true)
 |-- LatS: integer (nullable = true)
 |-- NS: string (nullable = true)
 |-- LonD: integer (nullable = true)
 |-- LonM: integer (nullable = true)
 |-- LonS: integer (nullable = true)
 |-- EW: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)



StructType([StructField('LatD', IntegerType(), True), StructField('LatM', IntegerType(), True), StructField('LatS', IntegerType(), True), StructField('NS', StringType(), True), StructField('LonD', IntegerType(), True), StructField('LonM', IntegerType(), True), StructField('LonS', IntegerType(), True), StructField('EW', StringType(), True), StructField('City', StringType(), True), StructField('State', StringType(), True)])

In [4]:
## Columns and expressions

from pyspark.sql.functions import col, column, expr

### Columns

col("some_column_name")
column("some_column_name")

df.select(col("EW"))
df["EW"]

### Expressions

df.select("LatD", expr("LatD - 5"), col("LatD") - 5, expr("LatD") - 5).limit(5).show()
df.columns

+----+----------+----------+----------+
|LatD|(LatD - 5)|(LatD - 5)|(LatD - 5)|
+----+----------+----------+----------+
|  41|        36|        36|        36|
|  42|        37|        37|        37|
|  46|        41|        41|        41|
|  42|        37|        37|        37|
|  43|        38|        38|        38|
+----+----------+----------+----------+



['LatD', 'LatM', 'LatS', 'NS', 'LonD', 'LonM', 'LonS', 'EW', 'City', 'State']

In [5]:
## Records and Rows

df.first()
df.first()

from pyspark.sql import Row

myRow = Row("test_row", None, False)
myRow[0], myRow[1], myRow[2]

('test_row', None, False)

In [8]:
## DataFrames transformation

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql import Row

schema = StructType([
    StructField("name", StringType(), True),
    StructField("language", StringType(), True),
    StructField("id", StringType(), True),
    StructField("bio", StringType(), True),
    StructField("version", FloatType(), True)
])

df = spark.read.format("json").option("multiLine", "true").schema(schema).load("data/sample.json")
df.show()

customSchema = StructType([ StructField("greet", StringType(), True), StructField("age", IntegerType(), True), StructField("num", IntegerType(), False) ])
customRow = Row("Hello", None, 1)
customDf = spark.createDataFrame([customRow], customSchema)
customDf.show()

df.createOrReplaceTempView("dfTable")
df = spark.sql("SELECT * FROM dfTable LIMIT 5")
df.show()

df.select("id","name").show()
df.select("id", col("id"), expr("id"), column("id"), "id" ).show()

df.select("id", col("id").alias("id_2"), expr("id").alias("id_3"), column("id").alias("id_4"), "id" ).show()

df.selectExpr("id as Id_1", "id as Id_2", "id as Id_3").show()

df.selectExpr("*", " (version < 5) as is_minus_than_five").show()

df.selectExpr("avg(version) as version_average", "count(distinct(name)) as distinct_name_counting").show()

+-----------------+----------------+----------------+--------------------+-------+
|             name|        language|              id|                 bio|version|
+-----------------+----------------+----------------+--------------------+-------+
|    Adeel Solangi|          Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|    Afzal Ghaffar|          Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|    Aamir Solangi|          Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|    Abla Dilmurat|          Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|         Adil Eli|          Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
|      Adile Qadir|          Uyghur|F2KEU5L7EHYSYFTT|Duis commodo orci...|    1.9|
|Abdukerim Ibrahim|          Uyghur|LO6DVTZLRK68528I|Vivamus id faucib...|    5.9|
|        Adil Abro|          Sindhi|LJRIULRNJFCNZJAJ|Etiam malesuada b...|   9.32|
| Afonso Vilarchán|        Galician|JMCL0CXNXHPL1GBC|Fusce eu ultrices...|   5.21|
|   

In [22]:
## Converting to Spark Types ( Literals )
from pyspark.sql.functions import lit

df.select("*", lit(1).alias("One")).show(2)

+-------------+--------+----------------+--------------------+-------+---+
|         name|language|              id|                 bio|version|One|
+-------------+--------+----------------+--------------------+-------+---+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|  1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|  1|
+-------------+--------+----------------+--------------------+-------+---+
only showing top 2 rows



In [23]:
# Adding Columns

df.withColumn("One", lit(1)).show(2)
df.withColumn("is_minus_than_five", expr("version < 5")).show(2)

+-------------+--------+----------------+--------------------+-------+---+
|         name|language|              id|                 bio|version|One|
+-------------+--------+----------------+--------------------+-------+---+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|  1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|  1|
+-------------+--------+----------------+--------------------+-------+---+
only showing top 2 rows

+-------------+--------+----------------+--------------------+-------+------------------+
|         name|language|              id|                 bio|version|is_minus_than_five|
+-------------+--------+----------------+--------------------+-------+------------------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|             false|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|              true|
+-------------+--------+----------------+--------------------+-------+-----

In [24]:
## Renaming Columns

df.withColumnRenamed("bio", "biography").show(2)

df.withColumnRenamed("name", "new-name").withColumnRenamed("new-name", "surname").show(2)

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|           biography|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
+-------------+--------+----------------+--------------------+-------+
only showing top 2 rows

+-------------+--------+----------------+--------------------+-------+
|      surname|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
+-------------+--------+----------------+--------------------+-------+
only showing top 2 rows



In [26]:
## Removing Columns

df.drop("bio").show(2)
df.drop("bio", "language").show(2)

+-------------+--------+----------------+-------+
|         name|language|              id|version|
+-------------+--------+----------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|   1.88|
+-------------+--------+----------------+-------+
only showing top 2 rows

+-------------+----------------+-------+
|         name|              id|version|
+-------------+----------------+-------+
|Adeel Solangi|V59OF92YF627HFY0|    6.1|
|Afzal Ghaffar|ENTOCR13RSCLZ6KU|   1.88|
+-------------+----------------+-------+
only showing top 2 rows



In [37]:
## Changing a column's type (cast)

casted_df = df.withColumn("version", col("version").cast("string") )
casted_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- language: string (nullable = true)
 |-- id: string (nullable = true)
 |-- bio: string (nullable = true)
 |-- version: string (nullable = true)



In [54]:
## Filtering Rows

from pyspark.sql.functions import length

df.show()

df.filter( col("version")>5 ).filter( col("language") == "Sindhi" ).show(2)
df.where( "version > 5 AND language = 'Sindhi'" ).show(2)

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
+-------------+--------+----------------+--------------------+-------+

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
+----

In [57]:
## Getting Unique Rows

df.select("language").distinct().show()
df.select("language").distinct().count()

+--------+
|language|
+--------+
|  Sindhi|
|  Uyghur|
+--------+



2

In [70]:
## Random Samples

df.sample(withReplacement=False, fraction=0.5, seed=2).show()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
+-------------+--------+----------------+--------------------+-------+



In [83]:
random_df = df.randomSplit([1.0, 10.0], seed=5)
random_df[0].show()
random_df[1].show()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
+-------------+--------+----------------+--------------------+-------+

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
+-------------+--------+----------------+--------------------+-------+



In [93]:
## Concatening and Appending Rows

from pyspark.sql.types import StructField, StructType, FloatType, StringType

new_schema = StructType([
    StructField('name', StringType(), True), 
    StructField('language', StringType(), True), 
    StructField('id', StringType(), False), 
    StructField('bio', StringType(), True),
    StructField('version', FloatType(), False)
])
row = ['Marko', 'Sindhi', 'QPOY123901HS90823HUURY9W', 'i dont kno', 1.3]
new_df = spark.createDataFrame([row], new_schema)

df.show()
new_df.show()

union_df = df.union(new_df)
union_df.where("language = 'Sindhi' and version < 5").show()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
+-------------+--------+----------------+--------------------+-------+

+-----+--------+--------------------+----------+-------+
| name|language|                  id|       bio|version|
+-----+--------+--------------------+----------+-------+
|Marko|  Sindhi|QPOY123901HS90823...|i dont kno|    1.3|
+-----+--------+--------------------+----------+-------+

+-------------+--------+--------------------+--------------------+-------+

In [101]:
## Sorting Rows

from pyspark.sql.functions import asc, desc

df.show()
df.sort("version").show()
df.sort("language", "version").show()

df.orderBy("version").show()
df.orderBy("language", col("version").desc() ).show()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
+-------------+--------+----------------+--------------------+-------+

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|Adee

In [105]:
## Limit

df.limit(5).show()
df.orderBy("version").limit(3).show()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
+-------------+--------+----------------+--------------------+-------+

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|Adee

In [118]:
## Repartition and Coalesce

df.rdd.getNumPartitions()
df.repartition(2)
df.rdd.getNumPartitions()

df.repartition(col('name'))
df.repartition(5, col('name'))

df.repartition(col('name')).coalesce(2)
df.repartition(5, col('name')).coalesce(2)

DataFrame[name: string, language: string, id: string, bio: string, version: float]

In [124]:
## Collecting Rows to the Driver

collectdf = df.limit(10)
collectdf.take(3)
collectdf.show()
collectdf.collect()

+-------------+--------+----------------+--------------------+-------+
|         name|language|              id|                 bio|version|
+-------------+--------+----------------+--------------------+-------+
|Adeel Solangi|  Sindhi|V59OF92YF627HFY0|Donec lobortis el...|    6.1|
|Afzal Ghaffar|  Sindhi|ENTOCR13RSCLZ6KU|Aliquam sollicitu...|   1.88|
|Aamir Solangi|  Sindhi|IAKPO3R4761JDRVG|Vestibulum pharet...|   7.27|
|Abla Dilmurat|  Uyghur|5ZVOEPMJUI4MB4EN|Donec lobortis el...|   2.53|
|     Adil Eli|  Uyghur|6VTI8X6LL0MMPJCC|Vivamus id faucib...|   6.49|
+-------------+--------+----------------+--------------------+-------+



[Row(name='Adeel Solangi', language='Sindhi', id='V59OF92YF627HFY0', bio='Donec lobortis eleifend condimentum. Cras dictum dolor lacinia lectus vehicula rutrum. Maecenas quis nisi nunc. Nam tristique feugiat est vitae mollis. Maecenas quis nisi nunc.', version=6.099999904632568),
 Row(name='Afzal Ghaffar', language='Sindhi', id='ENTOCR13RSCLZ6KU', bio='Aliquam sollicitudin ante ligula, eget malesuada nibh efficitur et. Pellentesque massa sem, scelerisque sit amet odio id, cursus tempor urna. Etiam congue dignissim volutpat. Vestibulum pharetra libero et velit gravida euismod.', version=1.8799999952316284),
 Row(name='Aamir Solangi', language='Sindhi', id='IAKPO3R4761JDRVG', bio='Vestibulum pharetra libero et velit gravida euismod. Quisque mauris ligula, efficitur porttitor sodales ac, lacinia non ex. Fusce eu ultrices elit, vel posuere neque.', version=7.269999980926514),
 Row(name='Abla Dilmurat', language='Uyghur', id='5ZVOEPMJUI4MB4EN', bio='Donec lobortis eleifend condimentum. Morb